## Ingest circuits.csv file
1. Read the file using Spark DataFrame Reader API.
1. Add Metadata Columns.
    - Source File
    - Ingestion Timestamp
1. Write to Bronze Delta Table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

#### Step 1 - Read the CSV file using Spark DataFrame Reader API.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema = StructType([
    StructField('circuitId', StringType()),
    StructField('url', StringType()),
    StructField('circuitName', StringType()),
    StructField('lat', DoubleType()),
    StructField('long', DoubleType()),
    StructField('locality', StringType()),
    StructField('country', StringType())
])

In [0]:
circuits_df = (
    spark.read
        .format('csv')
        .option('header', 'true')
#       .option('inferSchema', 'true')
        .option('mode', 'FAILFAST')
        .schema(circuits_schema)
        .load(source_file)
)

#### Step 2 - Add Metadata Columns.
- Source File
- Ingestion Timestamp

In [0]:
circuits_final_df = add_ingestion_metadata(circuits_df)

#### Step 3 - Write to Bronze Delta Table

In [0]:
write_to_bronze(input_df = circuits_final_df, target_table = table_name, batch_id = v_batch_id)